# Bitcoin (CRYPTO)

In [2]:
import time
import requests
import pandas as pd

In [ ]:
BASE_URL = "https://api.binance.com"  # Spot base endpoint
KLINES_EP = "/api/v3/klines"

def get_klines_paginated(symbol: str, interval: str, start: str, end: str | None = None,
                         limit: int = 1000, sleep: float = 0.25) -> pd.DataFrame:
    """
    Descarga klines paginando con startTime/endTime.
    - symbol: "BTCUSDT"
    - interval: "1m", "5m", "15m", "1h", ...
    - start/end: "YYYY-MM-DD" (end opcional; si None, llega hasta lo más reciente)
    """
    start_ms = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    end_ms = None if end is None else int(pd.Timestamp(end, tz="UTC").timestamp() * 1000)

    rows = []
    cur = start_ms

    while True:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": cur,
            "limit": min(limit, 1000)  # max 1000 según docs
        }
        if end_ms is not None:
            params["endTime"] = end_ms

        r = requests.get(BASE_URL + KLINES_EP, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        if not data:
            break

        rows.extend(data)

        # Cada kline: [openTime, open, high, low, close, volume, closeTime, ...]
        last_open_time = data[-1][0]
        next_cur = last_open_time + 1  # avanzar 1 ms para evitar duplicar la última vela

        if next_cur == cur:
            break
        cur = next_cur

        # Si pediste end y ya pasaste
        if end_ms is not None and cur >= end_ms:
            break

        time.sleep(sleep)

    df = pd.DataFrame(rows, columns=[
        "open_time","open","high","low","close","volume",
        "close_time","quote_asset_volume","n_trades",
        "taker_buy_base","taker_buy_quote","ignore"
    ])

    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)

    num_cols = ["open","high","low","close","volume","quote_asset_volume","taker_buy_base","taker_buy_quote"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["symbol"] = symbol
    df["interval"] = interval
    return df.sort_values("open_time").reset_index(drop=True)

# Ejemplo: BTCUSDT a 15 minutos desde 2018 hasta hoy (20180101_20260112)
df_btc_15m = get_klines_paginated("BTCUSDT", "15m", start="2018-01-01")
print(df_btc_15m.shape)
print(df_btc_15m.head())

(281131, 14)
                  open_time      open      high       low     close  \
0 2018-01-01 00:00:00+00:00  13715.65  13715.65  13400.01  13556.15   
1 2018-01-01 00:15:00+00:00  13533.75  13550.87  13402.00  13521.12   
2 2018-01-01 00:30:00+00:00  13500.00  13545.37  13450.00  13470.41   
3 2018-01-01 00:45:00+00:00  13494.65  13690.87  13450.00  13529.01   
4 2018-01-01 01:00:00+00:00  13528.99  13571.74  13402.28  13445.63   

       volume                       close_time  quote_asset_volume  n_trades  \
0  123.616013 2018-01-01 00:14:59.999000+00:00        1.675545e+06      1572   
1   98.136430 2018-01-01 00:29:59.999000+00:00        1.321757e+06      1461   
2   79.904037 2018-01-01 00:44:59.999000+00:00        1.078825e+06      1000   
3  141.699719 2018-01-01 00:59:59.999000+00:00        1.917783e+06      1195   
4   72.537533 2018-01-01 01:14:59.999000+00:00        9.778198e+05       898   

   taker_buy_base  taker_buy_quote ignore   symbol interval  
0       63.227133

In [ ]:
df_btc_15m.to_csv(
    r"C:\Users\cecil\Cript_Anomalies\BTCUSDT_15m_raw.csv",
    index=False
)